# Lotte Insight — 수집 파이프라인 (Colab GPU)

**실행 전 체크리스트**
- [ ] 런타임 → GPU (T4) 설정 확인
- [ ] Colab Secrets에 아래 키 등록
  - `SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY`
  - `NAVER_CLIENT_ID`, `NAVER_CLIENT_SECRET`
  - `OPENAI_API_KEY`
  - `HF_TOKEN` (private repo인 경우)
  - `HF_USERNAME` (HuggingFace 유저명)

In [ ]:
# ── Cell 1. GPU 확인 ──────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── Cell 2. 레포 클론 및 의존성 설치 ─────────────────────
!git clone https://github.com/<your-github-username>/lotte-insight.git
%cd lotte-insight/backend
!pip install -q -r requirements.txt
!pip install -q huggingface_hub

In [ ]:
# ── Cell 3. 환경변수 주입 (Colab Secrets) ────────────────
import os
from google.colab import userdata

os.environ['SUPABASE_URL']              = userdata.get('SUPABASE_URL')
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = userdata.get('SUPABASE_SERVICE_ROLE_KEY')
os.environ['NAVER_CLIENT_ID']           = userdata.get('NAVER_CLIENT_ID')
os.environ['NAVER_CLIENT_SECRET']       = userdata.get('NAVER_CLIENT_SECRET')
os.environ['OPENAI_API_KEY']            = userdata.get('OPENAI_API_KEY')

HF_USERNAME = userdata.get('HF_USERNAME')  # 예: 'joe123'
HF_TOKEN    = userdata.get('HF_TOKEN')     # private repo인 경우
if HF_TOKEN:
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

print('Environment variables set.')

In [ ]:
# ── Cell 4. 모델 다운로드 (HuggingFace Hub → /content/models/) ──
from huggingface_hub import snapshot_download

classifier_dir = snapshot_download(
    repo_id=f'{HF_USERNAME}/lotte-classifier-koelectra',
    local_dir='/content/models/classifier_koelectra',
)
summarizer_dir = snapshot_download(
    repo_id=f'{HF_USERNAME}/lotte-summarizer-kobart',
    local_dir='/content/models/summarizer_kobart',
)

os.environ['CLASSIFIER_MODEL_DIR'] = classifier_dir
os.environ['SUMMARIZER_MODEL_DIR'] = summarizer_dir

print('Classifier:', classifier_dir)
print('Summarizer:', summarizer_dir)

In [ ]:
# ── Cell 5. summarizer GPU 디바이스 패치 ─────────────────
# backend/models/summarizer.py 와 classifier.py 는 기본 CPU 로드
# GPU가 있으면 자동으로 cuda 디바이스를 사용하도록 패치
import sys
sys.path.insert(0, '/content/lotte-insight/backend')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Inference device:', DEVICE)

# 모듈 로드 전에 환경변수로 디바이스 힌트 전달
os.environ['INFERENCE_DEVICE'] = DEVICE

In [ ]:
# ── Cell 6. 파이프라인 실행 ───────────────────────────────
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    datefmt='%H:%M:%S',
)

from batch.news_collector import run
count = run()
print(f'\nProcessed articles: {count}')